# Volatility Surface Calibrator — From Option Prices to SVI

This notebook walks through the full pipeline:

1. **Black-Scholes recap** — from model price to implied volatility
2. **Fetching real market data** — live option chains via yfinance
3. **Computing implied volatility** — Brent's root-finding on BS
4. **The volatility smile** — why BS is wrong, and what the market tells us
5. **SVI model calibration** — fitting a parametric smile per maturity
6. **3D volatility surface** — assembling the full surface
7. **Arbitrage conditions** — what constraints must the surface satisfy?

In [ ]:
# Colab setup — run this cell if you are on Google Colab
import os
if "COLAB_RELEASE_TAG" in os.environ:
    !git clone --depth 1 https://github.com/louisgay/quant-apps.git /content/quant-apps
    !pip install -q -r /content/quant-apps/vol_surface_calibrator/requirements.txt
    os.chdir("/content/quant-apps/vol_surface_calibrator")

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import plotly.graph_objects as go

sys.path.insert(0, str(Path.cwd()))

from engine import (
    DataFetcher, compute_iv_chain,
    bs_call_price, bs_put_price, implied_volatility,
    SVISlice, SVICalibrator,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (12, 5), "font.size": 11})
print("Engine loaded.")

---
## 1. Black-Scholes Recap

The Black-Scholes call price is:

$$C(S,K,T,r,\sigma) = S \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$

where $d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}$ and $d_2 = d_1 - \sigma\sqrt{T}$.

Given a **market price**, we invert this formula to find the **implied volatility** $\sigma_{\text{impl}}$.

In [ ]:
# Demonstrate BS pricing and IV inversion
S, K, T, r = 100, 100, 1.0, 0.04

sigmas = np.linspace(0.05, 0.80, 200)
call_prices = [bs_call_price(S, K, T, r, s) for s in sigmas]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sigmas * 100, call_prices, linewidth=2, color="steelblue")
ax.set_xlabel("Volatility (%)")
ax.set_ylabel("Call Price (USD)")
ax.set_title("Black-Scholes Call Price as a Function of Volatility (ATM, T=1y)", fontweight="bold")

# Show inversion: given market price = 10, find IV
mkt_price = 10.0
iv = implied_volatility(mkt_price, S, K, T, r, "call")
ax.axhline(mkt_price, color="red", linestyle="--", alpha=0.7, label=f"Market price = ${mkt_price}")
ax.axvline(iv * 100, color="red", linestyle="--", alpha=0.7, label=f"IV = {iv*100:.1f}%")
ax.legend()
plt.tight_layout()
plt.show()
print(f"Market price ${mkt_price} => Implied Volatility = {iv:.4f} ({iv*100:.2f}%)")

---
## 2. Fetching Real Market Data

We use `yfinance` to pull live option chains for **SPY** (S&P 500 ETF). The data is cleaned by filtering on open interest, volume, and moneyness.

In [ ]:
TICKER = "SPY"

fetcher = DataFetcher(risk_free_rate=0.045)
data = fetcher.fetch(
    TICKER,
    min_oi=50,
    min_volume=5,
    max_expiries=6,
    moneyness_range=(0.85, 1.15),
)

print(f"Spot: ${data.spot:.2f}")
print(f"Options fetched: {data.n_options}")
print(f"Expiries: {data.expiries}")
data.chains.head(10)

---
## 3. Computing Implied Volatility

For each option in the chain, we solve:

$$\text{Find } \sigma \text{ such that } C^{BS}(S, K, T, r, \sigma) = C^{\text{market}}$$

using **Brent's root-finding algorithm** (`scipy.optimize.brentq`), which provides unconditional convergence on a bracketed interval.

In [ ]:
iv_df = compute_iv_chain(data.chains, data.spot, data.risk_free_rate)
print(f"Valid IVs: {len(iv_df)} / {data.n_options}")
iv_df[["strike", "expiry", "T", "option_type", "mid_price", "iv", "log_moneyness", "total_var"]].head(10)

---
## 4. The Volatility Smile

If Black-Scholes were correct, all options on the same underlying would have the **same** implied volatility regardless of strike. The market tells a different story.

The **skew** (IV increasing for lower strikes) reflects:
- **Leverage effect**: stocks fall faster than they rise
- **Crash risk premium**: OTM puts are expensive insurance
- **Supply/demand**: structured products create hedging flows in the wings

In [ ]:
expiries = sorted(iv_df["expiry"].unique())
n_exp = len(expiries)
cols = min(n_exp, 3)
rows = (n_exp + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)

for idx, exp in enumerate(expiries):
    ax = axes[idx // cols][idx % cols]
    grp = iv_df[iv_df["expiry"] == exp]
    T = grp["T"].iloc[0]

    calls = grp[grp["option_type"] == "call"]
    puts = grp[grp["option_type"] == "put"]

    ax.scatter(calls["strike"], calls["iv"] * 100, s=15, label="Calls", alpha=0.7)
    ax.scatter(puts["strike"], puts["iv"] * 100, s=15, marker="D", label="Puts", alpha=0.7)
    ax.axvline(data.spot, color="gray", linestyle="--", alpha=0.5, label="Spot")
    ax.set_title(f"{exp}  (T={T:.3f}y)", fontweight="bold")
    ax.set_xlabel("Strike")
    ax.set_ylabel("IV (%)")
    ax.legend(fontsize=8)

# Hide empty subplots
for idx in range(n_exp, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

fig.suptitle(f"Implied Volatility Smiles — {TICKER}", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 5. SVI Model Calibration

The **SVI** (Stochastic Volatility Inspired) model parameterises the total implied variance as:

$$w(k) = a + b \left[ \rho(k - m) + \sqrt{(k-m)^2 + \sigma^2} \right]$$

where $k = \ln(K/F)$ is log-moneyness and $w = \sigma_{\text{impl}}^2 \cdot T$ is total variance.

We calibrate this to each maturity slice by minimising the squared error, subject to the no-arbitrage constraint $a + b\sigma\sqrt{1-\rho^2} \geq 0$.

In [ ]:
calibrator = SVICalibrator()
surface = calibrator.calibrate_surface(iv_df, ticker=TICKER)

# Display parameters
display(surface.summary_df().style.format({
    "a": "{:.4f}", "b": "{:.4f}", "rho": "{:.3f}",
    "m": "{:.4f}", "sigma": "{:.4f}", "T": "{:.4f}",
    "rmse": "{:.2e}",
}))

In [ ]:
# Overlay SVI fit on market data
fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 4*rows), squeeze=False)

for idx, exp in enumerate(expiries):
    ax = axes[idx // cols][idx % cols]
    grp = iv_df[iv_df["expiry"] == exp]
    slc = surface.get_slice(exp)

    ax.scatter(grp["strike"], grp["iv"] * 100, s=15, color="gray",
               alpha=0.6, label="Market", zorder=5)

    if slc is not None:
        k_fine = np.linspace(grp["log_moneyness"].min() - 0.05,
                             grp["log_moneyness"].max() + 0.05, 300)
        fwd = grp["forward"].iloc[0]
        strikes_fine = fwd * np.exp(k_fine)
        iv_fit = slc.implied_vol(k_fine) * 100
        ax.plot(strikes_fine, iv_fit, color="#e74c3c", linewidth=2, label="SVI Fit")

    ax.axvline(data.spot, color="gray", linestyle="--", alpha=0.3)
    ax.set_title(f"{exp}  (RMSE={slc.rmse:.2e})" if slc else exp, fontweight="bold")
    ax.set_xlabel("Strike")
    ax.set_ylabel("IV (%)")
    ax.legend(fontsize=8)

for idx in range(n_exp, rows * cols):
    axes[idx // cols][idx % cols].set_visible(False)

fig.suptitle(f"SVI Fit vs Market IV — {TICKER}", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

---
## 6. 3D Volatility Surface

We assemble the full surface by interpolating total variance linearly across maturities (which preserves calendar-spread no-arbitrage by construction).

In [ ]:
k_grid = np.linspace(-0.30, 0.30, 80)
T_min = min(s.T for s in surface.slices)
T_max = max(s.T for s in surface.slices)
T_grid = np.linspace(T_min, T_max, 60)

iv_surface = surface.implied_vol_grid(k_grid, T_grid) * 100

fig = go.Figure(data=[
    go.Surface(
        x=k_grid, y=T_grid, z=iv_surface,
        colorscale="Viridis",
        colorbar=dict(title="IV (%)"),
        opacity=0.9,
    ),
    go.Scatter3d(
        x=iv_df["log_moneyness"],
        y=iv_df["T"],
        z=iv_df["iv"] * 100,
        mode="markers",
        marker=dict(size=2.5, color="red"),
        name="Market IV",
    ),
])

fig.update_layout(
    scene=dict(
        xaxis_title="Log-Moneyness (k)",
        yaxis_title="Time to Expiry (years)",
        zaxis_title="IV (%)",
        camera=dict(eye=dict(x=1.8, y=-1.5, z=0.8)),
    ),
    height=600, width=900,
    title=f"SVI Volatility Surface — {TICKER}",
)
fig.show()

---
## 7. Arbitrage Conditions

A vol surface is arbitrage-free if and only if three conditions hold simultaneously:

### 7.1 No Calendar-Spread Arbitrage
Total variance must be **non-decreasing in $T$** for every fixed strike:
$$\frac{\partial w(k, T)}{\partial T} \geq 0$$

Our linear interpolation of $w$ in $T$ guarantees this.

### 7.2 No Butterfly Arbitrage
The local volatility derived from the surface must be non-negative, which requires:
$$g(k) = \left(1 - \frac{k w'(k)}{2w(k)}\right)^2 - \frac{w'(k)^2}{4}\left(\frac{1}{w(k)} + \frac{1}{4}\right) + \frac{w''(k)}{2} \geq 0$$

### 7.3 SVI Slice-Level Condition
$$a + b \cdot \sigma \cdot \sqrt{1 - \rho^2} \geq 0$$

Let's verify our calibrated slices.

In [ ]:
print("Arbitrage-free check per slice:")
print("-" * 50)
for slc in surface.slices:
    lhs = slc.a + slc.b * slc.sigma * np.sqrt(1 - slc.rho**2)
    status = "PASS" if slc.is_arbitrage_free() else "FAIL"
    print(f"  {slc.expiry:12s}  a + b*sigma*sqrt(1-rho^2) = {lhs:+.6f}  [{status}]")

# Calendar check: total variance at ATM (k=0) must increase with T
print("\nCalendar-spread check (ATM total variance vs T):")
print("-" * 50)
prev_w = 0.0
for slc in sorted(surface.slices, key=lambda s: s.T):
    w_atm = float(slc.total_variance(np.array([0.0]))[0])
    status = "PASS" if w_atm >= prev_w - 1e-8 else "FAIL"
    print(f"  T={slc.T:.4f}  w(0)={w_atm:.6f}  [{status}]")
    prev_w = w_atm

---
## Summary

| Step | Method | Key insight |
|------|--------|-------------|
| Data | yfinance + OI/volume filters | Garbage in → garbage out; data quality is paramount |
| IV | Brent root-finding on BS | Monotonicity of BS in $\sigma$ guarantees unique solution |
| Smile | SVI calibration (SLSQP + DE) | 5 parameters capture the full smile shape per maturity |
| Surface | Linear $w$-interpolation | Total-variance interpolation prevents calendar arbitrage |
| Arbitrage | Gatheral condition | Necessary (not sufficient) for a tradeable surface |

### Production Extensions
- **SSVI** (Surface SVI): global parametrisation that enforces calendar + butterfly no-arbitrage jointly
- **Dividend-adjusted forward**: use listed dividend futures or discrete dividend models
- **Bid/ask surfaces**: calibrate separate surfaces for bid and ask to price with spread
- **Live streaming**: connect to Bloomberg DAPI or Refinitiv for real-time surface updates